# import

In [54]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Environnement

In [55]:
env = gym.make("LunarLander-v3")

#récupération de l'environnement et des actions
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# print(state_dim)
# print(action_dim)

# Test de l'agent aléatoire
state, info = env.reset(seed=42)
total_reward = 0.0
done = False
steps = 0

while not done:
    # l'agent choisit une action au hasard
    action = env.action_space.sample()
    
    # on applique l'action
    next_state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    # l'épisode est fini si le vaisseau s'écrase, atterrit terminated ou atteint la limite de temps truncated
    done = terminated or truncated

print(f"Score de l'agent aléatoire : {total_reward:.1f} (en {steps} étapes)")
env.close()



Score de l'agent aléatoire : -142.2 (en 100 étapes)


# Agent PPO

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()


        # ce réseau va permettre de donner une probabiltié de la prochaine action a réaliser selon l'env du vaisseau
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64), # prend nos 8 valeurs d'entrées et les envoie dans une couche de 64 neuronnes
            nn.Tanh(), # convertit nos donneés entre -1 et 1 pour une meilleur exploitation 
            nn.Linear(64, 64), # on refournit nos données déjà entrainé sur nos 64 neuronnes dnas 64 nouveaux neurones une couche de plus
            nn.Tanh(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1) # garantit que la somme des probabilités vaut 1
        )
        
        # sort une seule valeur estimation de l'action
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        

    #  cet fonction permet de choisir le chemin que la donnée va suivre dans notre réseau de neuronne
    def forward(self, state):

        action_probs = self.actor(state)
        state_value = self.critic(state)
        return action_probs, state_value

    

# vérification de la compilation du modèle

# mise en place du modèle
dummy_model = ActorCritic(state_dim, action_dim).to(device)

# création d'un faux état (a 8 chiffres) convertit pour etre comprehensible pour un tensor 
dummy_state = torch.tensor(np.zeros(state_dim), dtype=torch.float32).to(device)

# récupération de l'action de l'actor et de l'estimation du critic
dummy_probs, dummy_value = dummy_model(dummy_state)


print(f"probabilités de l'action sortie : {dummy_probs.detach().cpu().numpy()}")
print(f"valeur de l'action : {dummy_value.item():.4f}")

probabilités de l'action sortie : [0.27083224 0.25498244 0.23048638 0.24369888]
valeur de l'action : -0.0631


# Rollout buffer